In [32]:
from pathlib import Path
import pandas as pd

project = Path.cwd() / "project-d"
processed = project / "data" / "processed"

audit = pd.read_csv(processed / "eu_relevance_audit_100.csv")
print("Audit papers:", len(audit))
print(audit["relevance_decision"].value_counts(dropna=False))

Audit papers: 100
relevance_decision
NaN        66
Include    22
Exclude     8
Unclear     4
Name: count, dtype: int64


In [33]:
todo = audit[audit["relevance_decision"].isna()]

for i, row in todo.head(10).iterrows():
    print("=" * 70)
    print("ROW:", i)
    print("TITLE:", row["title"])
    print("YEAR:", row["year"], "| TYPE:", row["type"])
    print("ABSTRACT:", row["abstract"] if isinstance(row["abstract"], str) else "No abstract")

ROW: 34
TITLE: An objective overview of lifestyle and quality of life in the Spanish population: a quantification of nutri-indices and qualitative health nutritypes
YEAR: 2025 | TYPE: article
ABSTRACT: Introduction: Introduction: precision nutritional epidemiology studies require the development of initiatives to obtain objective population data that enable the implementation of health strategies. Therefore, the integration of health determinants and risk factors is essential to ensure the quality of life associated with nutrition through the use of new health indicators such as nutritypes and nutritional indices. Objective: the aim of the PLENUFAR 7 project was to design and promote the use of nutritypes and nutritional indices, integrating metabolic variables and nutritional markers of the Spanish population through nutritional education. Methodology: the project was developed in two phases. The first phase involved training and evaluating healthcare professionals. The second phase r

In [50]:
decisions = {
    94: ("Include", ""),
    95: ("Include", ""),
    96: ("Include", ""),
    97: ("Exclude", "Cultural heritage ML; education only mentioned"),
    98: ("Include", ""),
    99: ("Include", ""),
}

for row, (decision, reason) in decisions.items():
    audit.loc[row, "relevance_decision"] = decision
    audit.loc[row, "exclusion_reason"] = reason

audit.to_csv(processed / "eu_relevance_audit_100.csv", index=False)
print(audit["relevance_decision"].value_counts(dropna=False))

relevance_decision
Include    75
Exclude    19
Unclear     6
Name: count, dtype: int64


In [35]:
decisions = {
    24: ("Include", ""),
    25: ("Exclude", "AI analyses motor-competence data; education only a possible application"),
    26: ("Include", ""),
    27: ("Include", ""),
    28: ("Exclude", "Patient education in healthcare, not an education setting"),
}

In [51]:
todo = audit[audit["relevance_decision"].isna()]

for i, row in todo.head(10).iterrows():
    print("=" * 70)
    print("ROW:", i)
    print("TITLE:", row["title"])
    print("YEAR:", row["year"], "| TYPE:", row["type"])
    print("ABSTRACT:", row["abstract"] if isinstance(row["abstract"], str) else "No abstract")

In [52]:
import math

inc = (audit["relevance_decision"] == "Include").sum()
exc = (audit["relevance_decision"] == "Exclude").sum()
unc = (audit["relevance_decision"] == "Unclear").sum()

def wilson_ci(k, n, z=1.96):
    p = k / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return c - h, c + h

precision = inc / (inc + exc)
lo, hi = wilson_ci(inc, inc + exc)

results = pd.DataFrame([
    {"measure": "Precision (Unclear left out)", "value_%": round(100 * precision, 1),
     "ci95": f"{100*lo:.1f}–{100*hi:.1f}%"},
    {"measure": "Worst case (Unclear = Exclude)", "value_%": round(100 * inc / 100, 1), "ci95": ""},
    {"measure": "Best case (Unclear = Include)", "value_%": round(100 * (inc + unc) / 100, 1), "ci95": ""},
])

(processed / "validation").mkdir(exist_ok=True)
results.to_csv(processed / "validation" / "over_capture_results.csv", index=False)
print(results)

                          measure  value_%        ci95
0    Precision (Unclear left out)     79.8  70.6–86.7%
1  Worst case (Unclear = Exclude)     75.0            
2   Best case (Unclear = Include)     81.0            


In [53]:
excluded = audit[audit["relevance_decision"] == "Exclude"]
for i, row in excluded.iterrows():
    print(i, "|", row["title"][:70], "|", row["exclusion_reason"])

0 | Adaptive Artificial Intelligence Strategy for Multidimensional Dataset | Primarily general-purpose dataset evaluation
10 | Guest editorial: Generative AI and the government: looking through, th | Focuses on AI governance, not education
13 | Diabetes Monitoring System in Smart Health Cities Based on Big Data In | Focuses on healthcare and diabetes prediction
23 | GeoVectors-Central-America-location (v0.1) | OpenStreetMap dataset; 'education' only in funder name
25 | Modelling Motor Competence Interdependencies: An XAI and Graph Analyti | AI analyses motor-competence data; education only a possible application
28 | OP0036 Artificial Intelligence Reinforced Social Robotics in Patient C | Patient education in healthcare, not an education setting
30 | Farm Animal Welfare on the Mind and Meat in the Diet: Who Are These Co | Diet study; education only a demographic variable
33 | A digital online platform for education and certification of diabetic  | E-learning platform; ML only a side me

In [54]:
types = {
    "A. AI used elsewhere; education only possible use or mention": [25, 33, 45, 52, 68, 97],
    "B. Health or patient education": [13, 28, 34, 37],
    "C. Other AI topics, no education link": [0, 10, 54, 81],
    "D. 'Education' only in funder or ministry name": [23, 57, 71],
    "E. 'Education' only as a demographic variable": [30, 93],
}

for t, rows in types.items():
    audit.loc[rows, "exclusion_type"] = t

audit.to_csv(processed / "eu_relevance_audit_100.csv", index=False)

type_table = audit["exclusion_type"].value_counts().rename_axis("exclusion_type").reset_index(name="papers")
type_table.to_csv(processed / "validation" / "over_capture_exclusion_types.csv", index=False)
print(type_table)

                                      exclusion_type  papers
0  A. AI used elsewhere; education only possible ...       6
1              C. Other AI topics, no education link       4
2                     B. Health or patient education       4
3     D. 'Education' only in funder or ministry name       3
4      E. 'Education' only as a demographic variable       2


In [56]:
import requests

API_KEY = "PASTE_YOUR_KEY_HERE"

EU = "AT|BE|BG|HR|CY|CZ|DK|EE|FI|FR|DE|GR|HU|IE|IT|LV|LT|LU|MT|NL|PL|PT|RO|SK|SI|ES|SE"

r = requests.get("https://api.openalex.org/works", params={
    "filter": f"primary_topic.id:T14414,from_publication_date:2020-01-01,institutions.country_code:{EU}",
    "per-page": 1,
}, timeout=60)

print("Status:", r.status_code)
print("EU 'AI in Education' papers in OpenAlex:", r.json()["meta"]["count"])

Status: 200
EU 'AI in Education' papers in OpenAlex: 9512


In [57]:
r = requests.get("https://api.openalex.org/works", params={
    "filter": f"primary_topic.id:T14414,from_publication_date:2020-01-01,institutions.country_code:{EU}",
    "sample": 200,
    "seed": 42,
    "per-page": 200,
    "select": "id,title,publication_year,type",
}, timeout=60)
bench = pd.DataFrame(r.json()["results"])

collected = set(pd.read_csv(processed / "openalex_deduplicated.csv")["openalex_id"])
bench["found_by_our_search"] = bench["id"].isin(collected)

found = bench["found_by_our_search"].sum()
lo, hi = wilson_ci(found, len(bench))
print("Benchmark papers:", len(bench))
print("Found by our search:", found)
print(f"Recall: {100*found/len(bench):.1f}%  (95% CI {100*lo:.1f}–{100*hi:.1f}%)")

bench.to_csv(processed / "validation" / "under_capture_topic_sample.csv", index=False)

Benchmark papers: 200
Found by our search: 42
Recall: 21.0%  (95% CI 15.9–27.2%)


In [58]:
missed = bench[~bench["found_by_our_search"]]
for i, row in missed.head(20).iterrows():
    print(i, "|", row["publication_year"], "|", row["title"])

0 | 2026 | Sztuczna inteligencja w dydaktyce szkolnictwa wyższego: postawy, potrzeby i rekomendacje dla edukacji przyszłości
1 | 2025 | Perspectivas Iberoamericanas sobre la regulación de inteligencia artificial en educación superior: Un análisis comparativo
3 | 2025 | Fostering Creativity and Inclusion
5 | 2026 | Dataset: La inteligencia artificial generativa en la formación en Comunicación Digital: ¿qué percepciones y actitudes tiene el alumnado universitario?
6 | 2025 | Merkblatt zum Einsatz von Generativer KI an der HSLU-I
7 | 2026 | No me des la respuesta: protocolo de interacción guiada con inteligencia artificial generativa para el aprendizaje autónomo
8 | 2025 | Primary Teachers’ Use of AI: Practices, Professional Learning, and Integration Barriers
9 | 2025 | Experts’ Views on Generative AI’s Speculative Functions in Learning Design
11 | 2025 | Building and Utilizing a Specialized Chatbot in Physics Education – Experiment and Results
13 | 2025 | Underviserrettet AI i sprogfagen

In [59]:
r = requests.get("https://api.openalex.org/works", params={
    "filter": f"primary_topic.id:T14414,from_publication_date:2020-01-01,institutions.country_code:{EU}",
    "sample": 200, "seed": 42, "per-page": 200,
    "select": "id,language",
}, timeout=60)
lang = pd.DataFrame(r.json()["results"])
bench = bench.drop(columns="language", errors="ignore").merge(lang, on="id", how="left")
bench["english"] = bench["language"] == "en"

by_lang = bench.groupby("english")["found_by_our_search"].agg(papers="size", found="sum")
by_lang["recall_%"] = (100 * by_lang["found"] / by_lang["papers"]).round(1)
print(by_lang)

bench.to_csv(processed / "validation" / "under_capture_topic_sample.csv", index=False)

         papers  found  recall_%
english                         
False        83      9      10.8
True        117     33      28.2


In [60]:
missed_en = bench[bench["english"] & ~bench["found_by_our_search"]]
print("Missed English papers:", len(missed_en))
for i, row in missed_en.head(25).iterrows():
    print(i, "|", row["title"])

Missed English papers: 84
1 | Perspectivas Iberoamericanas sobre la regulación de inteligencia artificial en educación superior: Un análisis comparativo
3 | Fostering Creativity and Inclusion
9 | Experts’ Views on Generative AI’s Speculative Functions in Learning Design
11 | Building and Utilizing a Specialized Chatbot in Physics Education – Experiment and Results
17 | AI Text Generators and Text Producers
18 | Generative AI Tools in an Undergraduate Computer Science Program
22 | Passing the Turing Test: Fine-tuned AI feedback is less detectable than human or prompt-engineered feedback
24 | Ethical and Responsible Use of Generative Artificial Intelligence by University Students
28 | Creativity, Ethics, and AI
30 | Emerging human-technology relationships in a co-design process with generative AI
31 | Automated Essay Scoring with ChatGPT
32 | Examining students’ usage of artificial intelligence for UX projects : insights from an Informatics Service Management course
34 | Inteligencia art

In [61]:
terms = {
    "chatgpt": r"chatgpt",
    "chatbot": r"chat ?bot",
    "generative artificial intelligence": r"generative artificial intelligence",
    "genai": r"\bgenai\b",
    "LLM / large language model": r"large language model|\bllms?\b",
    "AI or artificial intelligence (any)": r"\bai\b|artificial intelligence",
}

missed_all = bench[~bench["found_by_our_search"]]
titles = missed_all["title"].fillna("").str.lower()

rows = []
for name, pat in terms.items():
    n = titles.str.contains(pat, regex=True).sum()
    rows.append({"term": name, "missed_papers_with_term": n,
                 "share_%": round(100 * n / len(missed_all), 1)})
term_table = pd.DataFrame(rows)
term_table.to_csv(processed / "validation" / "under_capture_missing_terms.csv", index=False)
print("Missed papers:", len(missed_all))
print(term_table)

Missed papers: 158
                                  term  missed_papers_with_term  share_%
0                              chatgpt                       13      8.2
1                              chatbot                        5      3.2
2   generative artificial intelligence                        6      3.8
3                                genai                        2      1.3
4           LLM / large language model                        4      2.5
5  AI or artificial intelligence (any)                       73     46.2


In [62]:
searches = {
    "ai_in_education": '"artificial intelligence in education"',
    "ai_education": '"AI in education"',
    "intelligent_tutoring": '"intelligent tutoring system"',
    "machine_learning_education": '"machine learning" education',
    "generative_ai_education": '"generative AI" education',
}

def count(search, extra=""):
    f = "from_publication_date:2020-01-01" + extra
    r = requests.get("https://api.openalex.org/works", params={
        "search.title_and_abstract": search, "filter": f, "per-page": 1}, timeout=60)
    r.raise_for_status()
    return r.json()["meta"]["count"]

rows = []
for name, term in searches.items():
    total = count(term)
    no_country = count(term, ",countries_distinct_count:0")
    rows.append({"search": name, "total": total, "no_country": no_country,
                 "no_country_%": round(100 * no_country / total, 1)})

aff = pd.DataFrame(rows)
aff.to_csv(processed / "validation" / "missing_affiliations_by_search.csv", index=False)
print(aff)

                       search  total  no_country  no_country_%
0             ai_in_education   4215        1578          37.4
1                ai_education   9547        3063          32.1
2        intelligent_tutoring   5915        1844          31.2
3  machine_learning_education  28765        7618          26.5
4     generative_ai_education  20831        6332          30.4


In [63]:
all_terms = " OR ".join(searches.values())

def by_year(extra=""):
    r = requests.get("https://api.openalex.org/works", params={
        "search.title_and_abstract": all_terms,
        "filter": "from_publication_date:2020-01-01" + extra,
        "group_by": "publication_year"}, timeout=60)
    r.raise_for_status()
    return {g["key"]: g["count"] for g in r.json()["group_by"]}

total = by_year()
missing = by_year(",countries_distinct_count:0")

year_table = pd.DataFrame({"total": total, "no_country": missing}).fillna(0).astype(int).sort_index()
year_table["no_country_%"] = (100 * year_table["no_country"] / year_table["total"]).round(1)
year_table.to_csv(processed / "validation" / "missing_affiliations_by_year.csv")
print(year_table)

       total  no_country  no_country_%
2020  340467      140948          41.4
2021  347479      130934          37.7
2022  330995      103736          31.3
2023  385175      121423          31.5
2024  447542      135028          30.2
2025  577751      180316          31.2
2026  549615      185081          33.7
2027      25          15          60.0
2028      11          10          90.9
2029       1           1         100.0
2030       1           1         100.0
2031       2           2         100.0


In [64]:
def by_year(term, extra=""):
    r = requests.get("https://api.openalex.org/works", params={
        "search.title_and_abstract": term,
        "filter": "from_publication_date:2020-01-01,to_publication_date:2026-12-31" + extra,
        "group_by": "publication_year"}, timeout=60)
    r.raise_for_status()
    return pd.Series({int(g["key"]): g["count"] for g in r.json()["group_by"]})

total = sum(by_year(t) for t in searches.values())
missing = sum(by_year(t, ",countries_distinct_count:0") for t in searches.values())

year_table = pd.DataFrame({"total": total, "no_country": missing}).fillna(0).astype(int).sort_index()
year_table["no_country_%"] = (100 * year_table["no_country"] / year_table["total"]).round(1)
year_table.to_csv(processed / "validation" / "missing_affiliations_by_year.csv")
print(year_table)

      total  no_country  no_country_%
2020   2002         626          31.3
2021   2413         569          23.6
2022   2947         654          22.2
2023   5155        1243          24.1
2024  10900        2714          24.9
2025  20201        5759          28.5
2026  25654        8869          34.6


In [65]:
summary = pd.DataFrame([
    ("Audit sample size", 100),
    ("Precision % (Unclear left out)", round(100 * precision, 1)),
    ("Precision 95% CI", f"{100*lo:.1f}–{100*hi:.1f}%"),
    ("Over-capture %", round(100 - 100 * precision, 1)),
    ("Recall % (OpenAlex AIED topic sample, n=200)", round(100 * found / len(bench), 1)),
    ("Recall % English papers", by_lang.loc[True, "recall_%"]),
    ("Recall % non-English papers", by_lang.loc[False, "recall_%"]),
    ("EU AIED-topic papers in OpenAlex", 9512),
    ("No-country % range across 5 searches", f"{aff['no_country_%'].min()}–{aff['no_country_%'].max()}%"),
    ("No-country % in 2026", year_table.loc[2026, "no_country_%"]),
], columns=["measure", "value"])

summary.to_csv(processed / "validation" / "validation_summary.csv", index=False)
print(summary)

                                        measure       value
0                             Audit sample size         100
1                Precision % (Unclear left out)        79.8
2                              Precision 95% CI  15.9–27.2%
3                                Over-capture %        20.2
4  Recall % (OpenAlex AIED topic sample, n=200)        21.0
5                       Recall % English papers        28.2
6                   Recall % non-English papers        10.8
7              EU AIED-topic papers in OpenAlex        9512
8          No-country % range across 5 searches  26.5–37.4%
9                          No-country % in 2026        34.6


In [67]:
p_lo, p_hi = wilson_ci(inc, inc + exc)
r_lo, r_hi = wilson_ci(found, len(bench))

summary.loc[summary["measure"] == "Precision 95% CI", "value"] = f"{100*p_lo:.1f}–{100*p_hi:.1f}%"
summary.loc[len(summary)] = ["Recall 95% CI", f"{100*r_lo:.1f}–{100*r_hi:.1f}%"]

summary.to_csv(processed / "validation" / "validation_summary.csv", index=False)
print(summary)

                                         measure       value
0                              Audit sample size         100
1                 Precision % (Unclear left out)        79.8
2                               Precision 95% CI  70.6–86.7%
3                                 Over-capture %        20.2
4   Recall % (OpenAlex AIED topic sample, n=200)        21.0
5                        Recall % English papers        28.2
6                    Recall % non-English papers        10.8
7               EU AIED-topic papers in OpenAlex        9512
8           No-country % range across 5 searches  26.5–37.4%
9                           No-country % in 2026        34.6
10                                 Recall 95% CI  15.9–27.2%
11                                 Recall 95% CI  15.9–27.2%
